# Semana 3 — Pilas, colas, búsqueda y tablas hash aplicadas a Penonomé

Aplica las unidades 22 (pilas), 23 (colas), 24 (búsqueda secuencial), 25 (búsqueda binaria) y 26 (tablas hash) a los datos oficiales del proyecto.

[Proyecto](../../README.md) | [Anterior](s02-funciones-clausuras-clases.ipynb) | [Siguiente](s04-ordenamiento-greedy-divide-y-venceras.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Datos

Se usan cinco tablas del proyecto, leídas con el módulo `csv` mediante `penonome.carga`:

- `fuentes.csv`: todas las fuentes de la investigación, con su URL y su estado de revisión.
- `00_catalogo_corregimientos.csv`: los 16 corregimientos con su clave y la nota de cambio de límites.
- `01_censo2023_poblacion_corregimientos.csv`: superficie y población de los 16 corregimientos de Penonomé.
- `afectaciones_sinaproc_cocle.csv`: afectaciones del sismo del 9-oct-2026 en Coclé según la EDAN de SINAPROC, en formato largo (40 filas).
- `99_base_larga_todo.csv`: el Censo y el IPM del PNUD en formato largo, una fila por dato (460 filas).

Las estructuras (`Pila`, `Cola`, `TablaHash`) viven en `penonome.estructuras` y las búsquedas en `penonome.busqueda`. Los algoritmos son Python puro: no se usa pandas.

In [2]:
import inspect
from math import ceil, log2

from penonome.carga import (PADRES_LEY_296, cargar_afectaciones, cargar_base_larga, cargar_catalogo,
                            cargar_fuentes, cargar_poblacion)
from penonome.estructuras import Cola, Pila, TablaHash, clave_ascii
from penonome.busqueda import bin_search, bin_search_contando, find_largest, find_two, seq_search_contando

fuentes = cargar_fuentes()
poblacion = cargar_poblacion()
catalogo = cargar_catalogo()
base = cargar_base_larga()
afectaciones = cargar_afectaciones()

print("Fuentes:", len(fuentes), "| corregimientos:", len(poblacion), "| filas de la base larga:", len(base),
      "| afectaciones:", len(afectaciones))
import csv
from penonome import FUENTES
with open(FUENTES, encoding="utf-8-sig", newline="") as archivo:
    filas_csv = sum(1 for _ in csv.DictReader(archivo))
verificar("cargar_fuentes lee todas las filas de fuentes.csv", len(fuentes), filas_csv)
verificar("Penonomé tiene 16 corregimientos", len(poblacion), 16)
verificar("la base larga tiene 460 filas", len(base), 460)
verificar("las afectaciones de Coclé tienen 40 filas", len(afectaciones), 40)

Fuentes: 113 | corregimientos: 16 | filas de la base larga: 460 | afectaciones: 40
Correcto: cargar_fuentes lee todas las filas de fuentes.csv
Correcto: Penonomé tiene 16 corregimientos
Correcto: la base larga tiene 460 filas
Correcto: las afectaciones de Coclé tienen 40 filas


## 1. Pila: deshacer las correcciones del 24-nov-2026

**Referencia:** [Unidad 22 — Pilas](../../../../01-python/semana-03/clase-1-pilas-colas-busqueda-secuencial/u22-pilas.ipynb). Una pila es LIFO (*Last In, First Out*): el último elemento que entra es el primero que sale. Es la estructura natural de un historial de cambios, porque deshacer revierte primero el cambio más reciente.

La clase `Pila` del proyecto es la `Stack` del curso con `peek` y `size`:

In [3]:
print(inspect.getsource(Pila))

class Pila:
    """Pila (LIFO): el último elemento que entra es el primero que sale."""

    def __init__(self):
        self.stack = []

    def is_empty(self):
        return len(self.stack) == 0

    def push(self, item):
        self.stack.append(item)

    def pop(self):
        return None if self.is_empty() else self.stack.pop()

    def peek(self):
        return None if self.is_empty() else self.stack[-1]

    def size(self):
        return len(self.stack)



El LEEME del paquete enumera las correcciones del 24-nov-2026. La que sigue vigente es la de las notas de los corregimientos divididos: en `00` y `99` decían «dividido en 2023» y ahora citan la Ley 296 de 2022. Se reconstruye el estado anterior de las cinco notas del catálogo y se aplican en orden; cada corrección se empuja (`push`) a una pila de historial con el valor anterior. El valor corregido se lee del archivo actual, de modo que al terminar el registro debe coincidir con los datos publicados.

In [4]:
def buscar_fila(filas, **condiciones):
    """Primera fila cuyas columnas coinciden con todas las condiciones (búsqueda secuencial)."""
    for fila in filas:
        if all(fila[columna] == valor for columna, valor in condiciones.items()):
            return fila
    return None

# Los cinco corregimientos que la Ley 296 dividió; su nota actual está en el catálogo
divididos = [buscar_fila(catalogo, id_corregimiento=clave) for clave in PADRES_LEY_296]

# Estado anterior a la verificación, según el LEEME (sección "Correcciones del 24-nov-2026")
registro = {f"00 nota de {fila['corregimiento']}": "dividido en 2023" for fila in divididos}
original = dict(registro)

# Valor corregido de cada dato, leído del archivo actual
correcciones = [(f"00 nota de {fila['corregimiento']}", fila["nota_cambio_limites"]) for fila in divididos]

historial = Pila()

def aplicar(registro, historial, campo, nuevo):
    historial.push((campo, registro[campo]))      # se guarda el valor anterior
    registro[campo] = nuevo

def deshacer(registro, historial):
    cambio = historial.pop()
    if cambio is None:
        return None                               # no hay nada que deshacer
    campo, anterior = cambio
    registro[campo] = anterior
    return campo

for campo, nuevo in correcciones:
    aplicar(registro, historial, campo, nuevo)

print("Cambios en la pila:", historial.size(), "| cima:", historial.peek()[0])
print("Nota corregida de Pajonal:", registro["00 nota de Pajonal"])
verificar("la pila guarda las cinco correcciones", historial.size(), 5)
verificar("la cima es la última corrección aplicada", historial.peek()[0], correcciones[-1][0])
verificar("cada nota corregida cita la Ley 296", all("Ley 296" in nota for nota in registro.values()), True)

Cambios en la pila: 5 | cima: 00 nota de Tulú
Nota corregida de Pajonal: dividido por la Ley 296 de 26-abr-2022 (corregimientos nuevos)
Correcto: la pila guarda las cinco correcciones
Correcto: la cima es la última corrección aplicada
Correcto: cada nota corregida cita la Ley 296


La corrección se justifica con una suma de control: las notas de los corregimientos divididos deben citar la Ley 296, y la base larga (tabla 99) debe decir lo mismo que el catálogo. Se cuentan las notas que no citan la ley con el valor corregido y, después de deshacer, con el valor anterior. Cada corregimiento dividido tiene tres filas con nota en la base larga (2000, 2010 y 2023).

In [5]:
def sin_citar_la_ley(registro):
    """Cuántas notas del registro no citan la Ley 296."""
    return sum(1 for nota in registro.values() if "Ley 296" not in nota)

filas_99 = [fila for fila in base if fila["nota"].startswith("dividido")]
nota_corregida = dict(correcciones)
coinciden_99 = all(fila["nota"] == nota_corregida[f"00 nota de {fila['corregimiento']}"] for fila in filas_99)
sin_ley_corregido = sin_citar_la_ley(registro)
print("Notas sin la ley con el valor corregido:", sin_ley_corregido, "| filas de la base larga con nota:", len(filas_99))

# Deshacer: el primer cambio revertido es el último aplicado (LIFO)
revertidos = [deshacer(registro, historial) for _ in range(len(correcciones) - 1)]
print("Orden en que se deshace:", revertidos)
print("Cima tras deshacer cuatro cambios:", historial.peek()[0])

verificar("se deshace en orden inverso al aplicado", revertidos, [c[0] for c in reversed(correcciones[1:])])
verificar("la cima es ahora la primera corrección", historial.peek()[0], correcciones[0][0])
deshacer(registro, historial)
print("Notas sin la ley con el valor anterior:", sin_citar_la_ley(registro))
verificar("tras deshacer todo vuelve el estado original", registro, original)
verificar("deshacer con la pila vacía devuelve None", deshacer(registro, historial), None)
verificar("con el valor anterior ninguna de las cinco notas cita la ley", sin_citar_la_ley(registro), len(correcciones))
verificar("con el valor corregido todas la citan", sin_ley_corregido, 0)
verificar("la base larga tiene tres filas con nota por corregimiento dividido", len(filas_99), 3 * len(correcciones))
verificar("las notas de la base larga coinciden con las del catálogo", coinciden_99, True)

Notas sin la ley con el valor corregido: 0 | filas de la base larga con nota: 15
Orden en que se deshace: ['00 nota de Tulú', '00 nota de Toabré', '00 nota de Río Indio', '00 nota de Pajonal']
Cima tras deshacer cuatro cambios: 00 nota de Chiguirí Arriba
Correcto: se deshace en orden inverso al aplicado
Correcto: la cima es ahora la primera corrección
Notas sin la ley con el valor anterior: 5
Correcto: tras deshacer todo vuelve el estado original
Correcto: deshacer con la pila vacía devuelve None
Correcto: con el valor anterior ninguna de las cinco notas cita la ley
Correcto: con el valor corregido todas la citan
Correcto: la base larga tiene tres filas con nota por corregimiento dividido
Correcto: las notas de la base larga coinciden con las del catálogo


### La pila también valida las referencias

`check_parentheses` de la unidad 22 comprueba que cada apertura tenga su cierre del mismo tipo. Cada referencia de `fuentes.csv` es un enlace de Markdown `[texto](url)`; si falta un corchete o un paréntesis, el enlace se rompe en el documento. Se aplica la función del curso, sin cambios, a todas las referencias.

In [6]:
def check_parentheses(expr):
    opening = "[{("            # paréntesis de apertura
    closing = "]})"            # paréntesis de cierre, en la misma posición que su apertura
    stack = Pila()
    for char in expr:
        if char in opening:
            stack.push(char)
        elif char in closing:
            if stack.is_empty():
                return False                      # cierre con la pila vacía
            if opening.index(stack.pop()) != closing.index(char):
                return False                      # el cierre no coincide
    return stack.is_empty()                       # sobran abiertos si no está vacía

invalidas = [f["id"] for f in fuentes if not check_parentheses(f["referencia"])]
print("Referencias con signos desequilibrados:", invalidas)
print("Ejemplo roto:", check_parentheses("INEC. [Censo 2023](https://www.inec.gob.pa/archivos/x.pdf"))
verificar("todas las referencias están equilibradas", invalidas, [])
verificar("detecta un enlace sin cerrar", check_parentheses("[Censo 2023](https://inec.gob.pa"), False)

Referencias con signos desequilibrados: []
Ejemplo roto: False
Correcto: todas las referencias están equilibradas
Correcto: detecta un enlace sin cerrar


## 2. Cola: fuentes pendientes de verificación

**Referencia:** [Unidad 23 — Colas](../../../../01-python/semana-03/clase-1-pilas-colas-busqueda-secuencial/u23-colas.ipynb). Una cola es FIFO (*First In, First Out*): se atiende primero a quien llegó primero. Es el modelo de una lista de trabajo: las fuentes con estado "sin revisar" se encolan en el orden de su identificador y se revisan en ese mismo orden.

Regla de trabajo del ejemplo: una fuente oficial o de un organismo se coteja de inmediato; una fuente de prensa u otra se pasa a una segunda cola, la de revisión manual, que se atiende cuando la primera queda vacía (como en Josephus, el elemento no se descarta: vuelve a esperar al final). Se revisan 12 fuentes por jornada.

In [7]:
print(inspect.getsource(Cola))

class Cola:
    """Cola (FIFO): el primer elemento que entra es el primero que sale."""

    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0

    def enqueue(self, item):
        self.queue.append(item)

    def dequeue(self):
        return None if self.is_empty() else self.queue.pop(0)

    def peek(self):
        return None if self.is_empty() else self.queue[0]

    def size(self):
        return len(self.queue)



In [8]:
pendientes = Cola()
for fuente in fuentes:
    if fuente["estado"] == "sin revisar":
        pendientes.enqueue(fuente)

total_pendientes = pendientes.size()
manual = Cola()
orden_de_revision = []
POR_JORNADA = 12

while not pendientes.is_empty():
    fuente = pendientes.dequeue()
    if fuente["tipo"] in ("oficial", "organismo"):
        orden_de_revision.append(fuente["id"])    # se coteja ahora
    else:
        manual.enqueue(fuente)                    # espera en la segunda cola
primarias = len(orden_de_revision)
while not manual.is_empty():
    orden_de_revision.append(manual.dequeue()["id"])

jornadas = ceil(len(orden_de_revision) / POR_JORNADA)
print("Fuentes sin revisar:", total_pendientes, "| oficiales u organismos:", primarias,
      "| a revisión manual:", total_pendientes - primarias)
print("Primeras cinco en revisarse:", orden_de_revision[:5])
print("Jornadas necesarias a", POR_JORNADA, "por jornada:", jornadas)

ids_oficiales = [f["id"] for f in fuentes if f["estado"] == "sin revisar" and f["tipo"] in ("oficial", "organismo")]
verificar("la cola recibe todas las fuentes sin revisar", total_pendientes, sum(f["estado"] == "sin revisar" for f in fuentes))
verificar("las jornadas alcanzan y una menos no alcanzaría", (jornadas * POR_JORNADA >= total_pendientes, (jornadas - 1) * POR_JORNADA < total_pendientes), (True, True))
verificar("las oficiales se atienden en orden FIFO", orden_de_revision[:primarias], ids_oficiales)
verificar("ninguna fuente se pierde ni se repite", sorted(orden_de_revision),
          sorted(f["id"] for f in fuentes if f["estado"] == "sin revisar"))
verificar("ambas colas quedan vacías", (pendientes.is_empty(), manual.is_empty()), (True, True))

Fuentes sin revisar: 33 | oficiales u organismos: 10 | a revisión manual: 23
Primeras cinco en revisarse: ['F001', 'F002', 'F011', 'F012', 'F017']
Jornadas necesarias a 12 por jornada: 3
Correcto: la cola recibe todas las fuentes sin revisar
Correcto: las jornadas alcanzan y una menos no alcanzaría
Correcto: las oficiales se atienden en orden FIFO
Correcto: ninguna fuente se pierde ni se repite
Correcto: ambas colas quedan vacías


**Error común:** suponer que `dequeue` es barato. La `Queue` del curso usa `list.pop(0)`, que desplaza todos los elementos restantes: cuesta O(n) por operación. Con unas decenas de fuentes no se nota, pero con millones sí.

**Ampliación (no está en el PDF):** `collections.deque` implementa una cola de doble extremo con `append` y `popleft` en O(1). Es la opción para colas grandes en código de producción.

## 3. Búsqueda secuencial

**Referencia:** [Unidad 24 — Búsqueda secuencial](../../../../01-python/semana-03/clase-1-pilas-colas-busqueda-secuencial/u24-busqueda-secuencial.ipynb). La búsqueda secuencial recorre la lista elemento por elemento hasta encontrar el valor; no exige orden y en el peor caso hace n comparaciones. La misma idea recorre la lista para hallar el mayor (`find_largest`) y el mayor y el menor a la vez (`find_two`).

Se busca en `fuentes.csv` la URL del Cuadro 1 del Censo 2023 (viviendas), que usa la base larga, y una URL que el estudio no registra. Después se buscan el mayor y el menor entre los corregimientos (por población) y entre los niveles de daño de las viviendas evaluadas por SINAPROC.

In [9]:
print(inspect.getsource(seq_search_contando))

url_cuadro1 = buscar_fila(base, indicador="censo_sin_radio")["url"]    # Censo 2023, Cuadro 1 (viviendas)
url_ausente = "https://www.inec.gob.pa/archivos/cuadro-que-no-figura-en-el-estudio.pdf"    # no se registró
por_url = lambda f: f["url"]
pos, comparaciones = seq_search_contando(fuentes, url_cuadro1, clave=por_url)
print("Cuadro 1 en fuentes.csv:", fuentes[pos]["id"], "| comparaciones:", comparaciones)
pos_falta, comp_falta = seq_search_contando(fuentes, url_ausente, clave=por_url)
print("URL que no figura en fuentes.csv: posición", pos_falta, "| comparaciones:", comp_falta)

mayor = find_largest(poblacion, clave=lambda c: c["pob_2023"])
i_mayor, i_menor, comp_two = find_two(poblacion, clave=lambda c: c["pob_2023"])
print("Corregimiento más poblado:", poblacion[mayor]["corregimiento"], poblacion[mayor]["pob_2023"])
print("Menos poblado:", poblacion[i_menor]["corregimiento"], poblacion[i_menor]["pob_2023"],
      "| comparaciones de find_two:", comp_two)

niveles = [f for f in afectaciones if f["categoria"] == "Viviendas por nivel de daño"]
por_valor = lambda f: f["valor"]
i_mas, i_menos, comp_niveles = find_two(niveles, clave=por_valor)
print("Nivel de daño con más viviendas:", niveles[i_mas]["subcategoria"], niveles[i_mas]["valor"],
      "| con menos:", niveles[i_menos]["subcategoria"], niveles[i_menos]["valor"])

verificar("la fuente encontrada tiene la URL del Cuadro 1", fuentes[pos]["url"], url_cuadro1)
verificar("una URL ausente cuesta n comparaciones (peor caso)", (pos_falta, comp_falta), (-1, len(fuentes)))
verificar("el más poblado es la cabecera", poblacion[mayor]["corregimiento"], "Penonomé (cabecera)")
verificar("el menos poblado es Riecito", poblacion[i_menor]["corregimiento"], "Riecito")
verificar("find_two hace entre n-1 y 2(n-1) comparaciones", 15 <= comp_two <= 30, True)
verificar("la mayoría de las viviendas evaluadas tuvo daño parcial",
          niveles[find_largest(niveles, clave=por_valor)]["subcategoria"], "Parcial")
verificar("find_two coincide con find_largest en los niveles de daño", i_mas, find_largest(niveles, clave=por_valor))

def seq_search_contando(nums, x, clave=_identidad):
    """Como `seq_search`, pero devuelve `(posición, comparaciones)`."""
    comparaciones = 0
    for i in range(len(nums)):
        comparaciones += 1
        if x == clave(nums[i]):
            return i, comparaciones
    return -1, comparaciones

Cuadro 1 en fuentes.csv: F009 | comparaciones: 5
URL que no figura en fuentes.csv: posición -1 | comparaciones: 113
Corregimiento más poblado: Penonomé (cabecera) 25836
Menos poblado: Riecito 1315 | comparaciones de find_two: 30
Nivel de daño con más viviendas: Parcial 1750 | con menos: Menor 75
Correcto: la fuente encontrada tiene la URL del Cuadro 1
Correcto: una URL ausente cuesta n comparaciones (peor caso)
Correcto: el más poblado es la cabecera
Correcto: el menos poblado es Riecito
Correcto: find_two hace entre n-1 y 2(n-1) comparaciones
Correcto: la mayoría de las viviendas evaluadas tuvo daño parcial
Correcto: find_two coincide con find_largest en los niveles de daño


## 4. Búsqueda binaria

**Referencia:** [Unidad 25 — Búsqueda binaria](../../../../01-python/semana-03/clase-2-busqueda-binaria-tablas-hash/u25-busqueda-binaria.ipynb). Si la lista está ordenada, se compara con el elemento del medio y se descarta la mitad que no puede contener el valor. Cada comparación divide el rango a la mitad: el peor caso es ⌊log₂ n⌋ + 1 comparaciones.

Se ordenan los corregimientos por población de 2023 con `sorted()` (los algoritmos de ordenamiento se implementan en la semana 4) y se busca cada población con los dos métodos.

In [10]:
print(inspect.getsource(bin_search))

por_poblacion = sorted(poblacion, key=lambda c: c["pob_2023"])
habitantes = lambda c: c["pob_2023"]

total_seq = total_bin = 0
coinciden = True
for c in poblacion:
    p_seq, c_seq = seq_search_contando(por_poblacion, c["pob_2023"], clave=habitantes)
    p_bin, c_bin = bin_search_contando(por_poblacion, c["pob_2023"], clave=habitantes)
    coinciden = coinciden and p_seq == p_bin
    total_seq += c_seq
    total_bin += c_bin

n = len(por_poblacion)
print(f"n = {n}: secuencial {total_seq / n:.2f} comparaciones en promedio, binaria {total_bin / n:.2f}")
posicion = bin_search(por_poblacion, 8496, clave=habitantes)
print("8,496 habitantes:", por_poblacion[posicion]["corregimiento"], "| posición", posicion)

verificar("ambas búsquedas dan la misma posición", coinciden, True)
verificar("8,496 habitantes es General Victoriano Lorenzo", por_poblacion[posicion]["corregimiento"], "General Victoriano Lorenzo")
verificar("una población inexistente devuelve -1", bin_search(por_poblacion, 5000, clave=habitantes), -1)

def bin_search(nums, x, clave=_identidad):
    """Búsqueda binaria (unidad 25). `nums` debe estar ordenada por `clave` de menor a mayor."""
    low, high = 0, len(nums) - 1
    while low <= high:
        mid = (low + high) // 2
        valor = clave(nums[mid])
        if valor == x:
            return mid
        elif valor > x:
            high = mid - 1
        else:
            low = mid + 1
    return -1

n = 16: secuencial 8.50 comparaciones en promedio, binaria 3.38
8,496 habitantes: General Victoriano Lorenzo | posición 12
Correcto: ambas búsquedas dan la misma posición
Correcto: 8,496 habitantes es General Victoriano Lorenzo
Correcto: una población inexistente devuelve -1


Con 16 elementos la diferencia es pequeña. Con las 460 filas de la base larga, ordenadas por `valor`, la búsqueda binaria nunca supera 9 comparaciones (⌊log₂ 460⌋ + 1), mientras que la secuencial necesita en promedio más de doscientas.

**Error común:** aplicar búsqueda binaria a una lista sin ordenar. El algoritmo no falla con un error: devuelve -1 o una posición equivocada sin avisar.

In [11]:
valores = sorted(fila["valor"] for fila in base)
n = len(valores)
peor_seq = peor_bin = suma_seq = suma_bin = 0
correctas = True
for x in valores:
    p_seq, c_seq = seq_search_contando(valores, x)
    p_bin, c_bin = bin_search_contando(valores, x)
    correctas = correctas and valores[p_bin] == x      # con repetidos puede ser otra posición del mismo valor
    suma_seq, suma_bin = suma_seq + c_seq, suma_bin + c_bin
    peor_seq, peor_bin = max(peor_seq, c_seq), max(peor_bin, c_bin)

limite = int(log2(n)) + 1
print(f"n = {n}")
print(f"secuencial: promedio {suma_seq / n:7.1f} | peor caso {peor_seq}")
print(f"binaria:    promedio {suma_bin / n:7.1f} | peor caso {peor_bin} (cota log2: {limite})")
_, c_ausente = bin_search_contando(valores, -1)
verificar("la binaria encuentra cada valor", correctas, True)
verificar("la binaria no supera la cota logarítmica", peor_bin <= limite, True)
verificar("un valor ausente también respeta la cota", c_ausente <= limite, True)

n = 460
secuencial: promedio   230.1 | peor caso 460
binaria:    promedio     7.5 | peor caso 9 (cota log2: 9)
Correcto: la binaria encuentra cada valor
Correcto: la binaria no supera la cota logarítmica
Correcto: un valor ausente también respeta la cota


## 5. Tabla hash: índice de fuentes por URL

**Referencia:** [Unidad 26 — Tablas hash](../../../../01-python/semana-03/clase-2-busqueda-binaria-tablas-hash/u26-tablas-hash.ipynb). Una tabla hash convierte la clave en una posición con una función hash y va directo al cubo correspondiente. Se usa la función del PDF: la suma de los códigos ASCII del texto, módulo el tamaño de la tabla.

**Ampliación (no está en el PDF):** la `HashTable` del curso guarda solo valores y su `get` devuelve el cubo entero. `TablaHash` guarda pares `(clave, valor)` en cada cubo para que `get` devuelva el valor exacto aunque varias URL compartan cubo (encadenamiento), y cuenta las comparaciones hechas dentro del cubo.

In [12]:
print(inspect.getsource(TablaHash.hash))
print(inspect.getsource(TablaHash.put))
print(inspect.getsource(TablaHash.get))

indice = TablaHash(151)
for fuente in fuentes:
    indice.put(fuente["url"], fuente["id"])

print("URL indexadas:", len(indice), "| cubos con colisión:", indice.colisiones(),
      "| cubo más largo:", indice.cubo_mas_largo())
print("Clave ASCII de la URL del Cuadro 1:", clave_ascii(url_cuadro1), "-> cubo", indice.hash(url_cuadro1))
print("get(URL del Cuadro 1):", indice.get(url_cuadro1))

indice.comparaciones = 0
iguales = all(indice.get(f["url"]) == f["id"] for f in fuentes)
promedio_hash = indice.comparaciones / len(fuentes)
promedio_seq = sum(seq_search_contando(fuentes, f["url"], clave=lambda g: g["url"])[1] for f in fuentes) / len(fuentes)
print(f"Comparaciones promedio por consulta: tabla hash {promedio_hash:.2f} | secuencial {promedio_seq:.1f}")

verificar("cada URL devuelve su identificador", iguales, True)
verificar("la tabla hash y la búsqueda secuencial dan la misma fuente", indice.get(url_cuadro1), fuentes[pos]["id"])
verificar("una URL ausente devuelve None", indice.get(url_ausente), None)
verificar("la tabla hash compara mucho menos que la secuencial", promedio_hash < promedio_seq / 10, True)

    def hash(self, key):
        """Posición de la clave: `clave % size`; los textos se convierten antes con `clave_ascii`."""
        if isinstance(key, str):
            key = clave_ascii(key)
        return key % self.size

    def put(self, key, value):
        """Guarda el valor; si la clave ya estaba, reemplaza su valor."""
        bucket = self.table[self.hash(key)]
        for i, (clave, _) in enumerate(bucket):
            if clave == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))

    def get(self, key, default=None):
        """Devuelve el valor de la clave o `default` si no está."""
        for clave, valor in self.table[self.hash(key)]:
            self.comparaciones += 1
            if clave == key:
                return valor
        return default

URL indexadas: 113 | cubos con colisión: 23 | cubo más largo: 4
Clave ASCII de la URL del Cuadro 1: 5448 -> cubo 12
get(URL del Cuadro 1): F009
Comparaciones promedio

### Control de trazabilidad: ¿están registradas las URL de la base larga?

Cada fila de la base larga cita la URL de su cuadro oficial. El índice permite comprobar en tiempo casi constante si cada una de esas URL figura en `fuentes.csv`, la lista de referencias del estudio. Se compara el resultado con la búsqueda secuencial.

In [13]:
urls_base = sorted(set(fila["url"] for fila in base))
registradas = [u for u in urls_base if indice.get(u) is not None]
faltantes = [u for u in urls_base if indice.get(u) is None]
faltantes_seq = [u for u in urls_base if seq_search_contando(fuentes, u, clave=por_url)[0] == -1]

print("URL distintas en la base larga:", len(urls_base), "| registradas en fuentes.csv:", len(registradas),
      "| sin registrar:", len(faltantes))
for u in faltantes:
    print("  ", u)
verificar("la tabla hash y la búsqueda secuencial detectan las mismas faltantes", faltantes, faltantes_seq)
verificar("todas las URL de la base larga están registradas", len(faltantes), 0)
verificar("toda URL se clasifica una sola vez", len(registradas) + len(faltantes), len(urls_base))

URL distintas en la base larga: 3 | registradas en fuentes.csv: 3 | sin registrar: 0
Correcto: la tabla hash y la búsqueda secuencial detectan las mismas faltantes
Correcto: todas las URL de la base larga están registradas
Correcto: toda URL se clasifica una sola vez


### Colisiones según el tamaño de la tabla

Con pocas ranuras casi todas las URL chocan; al crecer la tabla, los cubos se acortan y `get` se acerca a una sola comparación. También se usa la tabla para contar cuántas filas tiene cada indicador de la base larga, y se compara con un diccionario de Python (que es una tabla hash ya implementada).

In [14]:
print(f"{'ranuras':>8} {'cubos con colisión':>19} {'cubo más largo':>15}")
tablas = {}
for tamano in (8, 31, 151, 307):
    tabla = TablaHash(tamano)
    for fuente in fuentes:
        tabla.put(fuente["url"], fuente["id"])
    tablas[tamano] = tabla
    print(f"{tamano:>8} {tabla.colisiones():>19} {tabla.cubo_mas_largo():>15}")

tabla_8 = tablas[8]

conteo = TablaHash(97)
for fila in base:
    conteo.put(fila["indicador"], conteo.get(fila["indicador"], 0) + 1)

referencia = {}
for fila in base:
    referencia[fila["indicador"]] = referencia.get(fila["indicador"], 0) + 1

print("Indicadores distintos:", len(conteo), "| filas de 'poblacion':", conteo.get("poblacion"))
verificar("con 8 ranuras los 8 cubos tienen colisión", tabla_8.colisiones(), 8)
verificar("el conteo coincide con el diccionario de Python",
          all(conteo.get(k) == v for k, v in referencia.items()), True)
verificar("la tabla tiene tantas claves como indicadores", len(conteo), len(referencia))
verificar("el conteo suma las 460 filas", sum(conteo.get(k) for k in referencia), len(base))

 ranuras  cubos con colisión  cubo más largo
       8                   8              20
      31                  29               7
     151                  23               4
     307                  18               3
Indicadores distintos: 25 | filas de 'poblacion': 36
Correcto: con 8 ranuras los 8 cubos tienen colisión
Correcto: el conteo coincide con el diccionario de Python
Correcto: la tabla tiene tantas claves como indicadores
Correcto: el conteo suma las 460 filas


### Consultar las afectaciones por clave compuesta

Las filas de `afectaciones_sinaproc_cocle.csv` no tienen un identificador: cada dato se distingue por tres columnas (`categoria`, `subcategoria` y `desagregacion`). Unidas con `|` forman una clave de texto que la función hash convierte en una posición, y `get` devuelve el valor sin recorrer las 40 filas.

**Error común:** si dos filas compartieran la clave, `put` reemplazaría la primera sin avisar. Por eso se comprueba antes que la clave identifica una sola fila.

In [15]:
clave_de = lambda f: f"{f['categoria']}|{f['subcategoria']}|{f['desagregacion']}"
claves = [clave_de(f) for f in afectaciones]

indice_afectaciones = TablaHash(31)
for fila in afectaciones:
    indice_afectaciones.put(clave_de(fila), fila["valor"])

print("Claves distintas:", len(set(claves)), "de", len(afectaciones), "filas | cubos con colisión:", indice_afectaciones.colisiones())
print("Viviendas con daño parcial:", indice_afectaciones.get("Viviendas por nivel de daño|Parcial|Total"))
print("Lesionados:", indice_afectaciones.get("Otras afectaciones|Lesionados|Total"))

verificar("cada fila tiene una clave distinta", len(set(claves)), len(afectaciones))
verificar("get devuelve el valor exacto", indice_afectaciones.get("Viviendas por nivel de daño|Parcial|Total"), 1750)
verificar("la tabla coincide con la búsqueda secuencial",
          all(indice_afectaciones.get(clave_de(f)) == f["valor"] for f in afectaciones), True)
verificar("una clave que no existe devuelve None", indice_afectaciones.get("Cobertura|Puentes|Total"), None)

Claves distintas: 40 de 40 filas | cubos con colisión: 12
Viviendas con daño parcial: 1750
Lesionados: 17
Correcto: cada fila tiene una clave distinta
Correcto: get devuelve el valor exacto
Correcto: la tabla coincide con la búsqueda secuencial
Correcto: una clave que no existe devuelve None


## 6. Comparación de complejidad

| Operación | Estructura | Complejidad | Medido en este notebook |
|---|---|---|---|
| `push`, `pop`, `peek` | `Pila` (lista, al final) | O(1) | 5 cambios aplicados y deshechos |
| `enqueue` / `dequeue` | `Cola` (lista, `pop(0)`) | O(1) / O(n) | todas las fuentes sin revisar, atendidas en orden |
| Búsqueda secuencial | lista sin orden | O(n) | una comparación por fuente para una URL ausente; 230.1 en promedio en 460 valores |
| Búsqueda binaria | lista ordenada | O(log n) | como máximo 9 comparaciones en 460 valores |
| `get` en tabla hash | cubos con encadenamiento | O(1) promedio, O(n) peor caso | pocas comparaciones por consulta con 151 ranuras |

**Clave:** la búsqueda binaria exige pagar antes el ordenamiento (O(n log n)); conviene cuando se busca muchas veces sobre los mismos datos. La tabla hash no exige orden, pero depende de una buena función hash y de un tamaño suficiente: con 8 ranuras sus cubos degeneran en listas y vuelve a ser una búsqueda secuencial.

## Conclusiones

- La pila modela el historial de correcciones: se aplicaron las cinco correcciones de notas del 24-nov-2026 (los corregimientos que dividió la Ley 296), se deshicieron en orden inverso y la suma de control confirma que las cinco notas citan la ley solo con el valor corregido. Las 15 filas con nota de la base larga coinciden con el catálogo.
- `check_parentheses` del curso confirma que todas las referencias de `fuentes.csv` tienen sus enlaces bien cerrados.
- La cola ordena el trabajo pendiente: las fuentes sin revisar se atienden en orden FIFO, primero las oficiales y de organismos, y las jornadas necesarias salen de dividir el total entre 12 revisiones por jornada.
- La búsqueda secuencial no necesita orden, pero su costo crece con n; la binaria, sobre los datos ordenados, nunca pasó de 9 comparaciones en 460 valores. La misma recorrida encuentra que el nivel de daño con más viviendas evaluadas por SINAPROC en Coclé es el parcial.
- La tabla hash con la función ASCII del PDF indexa las fuentes por URL y las afectaciones del sismo por una clave compuesta (40 claves distintas), y responde con muy pocas comparaciones por consulta cuando tiene ranuras suficientes.
- El control de trazabilidad con el índice confirma que las 3 URL citadas en la base larga (los Cuadros 1 y 4 del Censo 2023 y el cuadro de Coclé del IPM del PNUD) están registradas en `fuentes.csv`.

[Proyecto](../../README.md) | [Anterior](s02-funciones-clausuras-clases.ipynb) | [Siguiente](s04-ordenamiento-greedy-divide-y-venceras.ipynb)